In [8]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, desc, sum, avg
import time
from pyspark.sql.functions import rand, col, when, concat, lit
# Initialize
spark = SparkSession.builder.appName("SparkCompleteNotes").getOrCreate()


In [10]:
# 10 million rows
df = spark.range(0, 10000000)

# Random value column (0-999)
df = df.withColumn(
    "value",
    (rand() * 1000).cast("int")
)


In [11]:
optimized_df = df.filter(col("value") > 500).filter(col("id") < 5000000).select("id", "value")

In [12]:
optimized_df.explain()

== Physical Plan ==
*(1) Filter (isnotnull(value#11) AND ((value#11 > 500) AND (id#9L < 5000000)))
+- *(1) Project [id#9L, cast((rand(4170682656138141575) * 1000.0) as int) AS value#11]
   +- *(1) Range (0, 10000000, step=1, splits=4)




In [14]:
start_time = time.time()

count_uncached = optimized_df.count()

end_time = time.time()

print(
    f"1. Optimized Execution | Count: {count_uncached} | Time: {round(end_time - start_time, 4)} seconds"
)

1. Optimized Execution | Count: 2496035 | Time: 0.6814 seconds


In [15]:
optimized_df.cache()

DataFrame[id: bigint, value: int]

In [17]:
# Cache fill karo
optimized_df.count()

# Ab actual cached timing lo
start_time = time.time()

optimized_df.count()

end_time = time.time()

print(f"Cached Time: {end_time - start_time:.4f} sec")

26/06/13 06:17:50 WARN CacheManager: Asked to cache already cached data.


Cached Time: 0.5291 sec


In [19]:
start_time = time.time()

count_persisted = optimized_df.count()

end_time = time.time()

print(
    f"Persisted Execution | Count: {count_persisted} | Time: {round(end_time - start_time, 4)} seconds"
)

Persisted Execution | Count: 2496035 | Time: 0.5325 seconds


In [21]:
data = [
    (1, "Alice", 75000),
    (2, "Bob", 60000),
    (3, "Charlie", 85000),
    (4, "David", 50000)
]

columns = ["id", "name", "salary"]

df = spark.createDataFrame(data, columns)

df.show()

+---+-------+------+
| id|   name|salary|
+---+-------+------+
|  1|  Alice| 75000|
|  2|    Bob| 60000|
|  3|Charlie| 85000|
|  4|  David| 50000|
+---+-------+------+



In [22]:
def salary_grade(salary):
    if salary >= 80000:
        return "High"
    elif salary >= 60000:
        return "Medium"
    else:
        return "Low"

In [23]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

grade_udf = udf(salary_grade, StringType())

In [24]:
df_with_grade = df.withColumn(
    "grade",
    grade_udf(df.salary)
)

df_with_grade.show()

+---+-------+------+------+
| id|   name|salary| grade|
+---+-------+------+------+
|  1|  Alice| 75000|Medium|
|  2|    Bob| 60000|Medium|
|  3|Charlie| 85000|  High|
|  4|  David| 50000|   Low|
+---+-------+------+------+



In [25]:
spark.stop()

In [1]:
print("Hello world")

Hello world
